# Treatment Check: Is the Treatment Feed More Diverse?

For each ranking cycle, Δ = H(M) − H(K): the Shannon diversity of the left, center and right
categories in the treatment (M) feed minus that of the control (K) feed ranked at the same moment.
Δ is averaged per day (22 September to 18 October 2026) and the 27 daily means are tested
against 0. Days, not cycles, are the independent units: the ~48 cycles of a day rank largely the
same articles.

| level | ranks | what it represents |
|---|---|---|
| `top10` | 1–10 | the first ten political clusters |
| `top20` | 1–20 | the first twenty political clusters |
| `tier_end` | all political clusters | the whole political part of the feed |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg

DATA_DIR = Path('../data')
FIG_DIR = Path('../tables_figures')
READ = dict(sep=';', decimal=',', encoding='utf-8-sig')
LEVELS = ['top10', 'top20', 'tier_end']
INK = '#333333'

# APA style: sans-serif, no title in the image, no top/right axis lines.
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'DejaVu Sans'],
                     'font.size': 10, 'axes.labelsize': 11, 'axes.edgecolor': INK,
                     'axes.labelcolor': INK, 'xtick.color': INK, 'ytick.color': INK,
                     'axes.spines.top': False, 'axes.spines.right': False})

rows = pd.read_csv(DATA_DIR / 'treatment_check_rankings_round1.csv', **READ)        # per cycle, arm and level
clusters = pd.read_csv(DATA_DIR / 'treatment_check_clusters_round1.csv', **READ)    # political clusters, end-of-day cycles
pushed = pd.read_csv(DATA_DIR / 'treatment_check_sources_round1.csv', **READ)       # served articles, end-of-day cycles
parties = pd.read_csv(DATA_DIR / 'party_positions_ches2024.csv', **READ)
print(f"{rows['cycle_id'].nunique()} ranking cycles over {rows['day'].nunique()} days")

## Figure 1: party positions and ideology categories

In [ ]:
def category(lrgen):
    """Left below 4, center from 4 to 6, right above 6 on the CHES 0-10 scale."""
    return 'left' if lrgen < 4 else 'center' if lrgen <= 6 else 'right'


parties['lrgen_cat'] = parties['lrgen'].map(category)
parties.sort_values('lrgen').round(2)

In [ ]:
# Category colours. Deliberately NOT the national red/blue convention, which is
# inverted between Europe and the US and would mislead half the readership.
COLOURS = {"left": "#D1495B", "center": "#EDAE49", "right": "#00798C"}

W, H = 1180, 400          # canvas
M_L, M_R = 70, 70         # horizontal margins
BASE_Y = 300              # the 0-10 axis line
LANE_H = 58               # vertical offset per stacked point
R = 13                    # dot radius
# Minimum horizontal gap, in lrgen units, between two labels sharing a lane.
# 0.75 units ~= 78px here, which clears the widest party label at 16px.
NEAR = 0.75

In [ ]:
def x_of(lrgen: float) -> float:
    return M_L + (lrgen / 10.0) * (W - M_L - M_R)


def esc(s: str) -> str:
    return (s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))


def build(parties: dict) -> str:
    rows = sorted(
        ({"party": k, "lrgen": float(v["lrgen"]), "cat": str(v.get("lrgen_cat", ""))}
         for k, v in parties.items() if "lrgen" in v),
        key=lambda r: r["lrgen"],
    )
    # Greedy lane packing: take the LOWEST lane whose occupants are all farther than
    # NEAR away. Counting nearby predecessors (the obvious approach) does not work --
    # it can assign two points the same lane while they still overlap, which put DeFI
    # and Anders on top of each other in an earlier draft.
    lanes: list[list[float]] = []
    for r in rows:
        for idx, occupants in enumerate(lanes):
            if all(abs(x - r["lrgen"]) > NEAR for x in occupants):
                occupants.append(r["lrgen"])
                r["lane"] = idx
                break
        else:
            lanes.append([r["lrgen"]])
            r["lane"] = len(lanes) - 1

    out: list[str] = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" '
        f'viewBox="0 0 {W} {H}" font-family="Arial, Helvetica, sans-serif">',
        f'<rect width="{W}" height="{H}" fill="#ffffff"/>',
    ]

    # Centre line at 5, drawn first so dots sit on top.
    out.append(f'<line x1="{x_of(5):.1f}" y1="{BASE_Y - 250:.1f}" x2="{x_of(5):.1f}" '
               f'y2="{BASE_Y:.1f}" stroke="#999" stroke-width="1" stroke-dasharray="4 4"/>')
    # Axis.
    out.append(f'<line x1="{M_L}" y1="{BASE_Y}" x2="{W - M_R}" y2="{BASE_Y}" '
               f'stroke="#111" stroke-width="1.4"/>')
    for t in range(11):
        x = x_of(t)
        out.append(f'<line x1="{x:.1f}" y1="{BASE_Y}" x2="{x:.1f}" y2="{BASE_Y + 7}" '
                   f'stroke="#111" stroke-width="1.1"/>')
        out.append(f'<text x="{x:.1f}" y="{BASE_Y + 28}" text-anchor="middle" '
                   f'font-size="15" fill="#111">{t}</text>')
    out.append(f'<text x="{W/2:.1f}" y="{BASE_Y + 58}" text-anchor="middle" font-size="16" '
               f'fill="#111">CHES general left–right position (0 = left, 10 = right)</text>')

    # Points, bottom lane upward.
    for r in rows:
        x, y = x_of(r["lrgen"]), BASE_Y - 34 - r["lane"] * LANE_H
        fill = COLOURS.get(r["cat"], "#777")
        out.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{R}" fill="{fill}" '
                   f'stroke="#111" stroke-width="1.3"/>')
        out.append(f'<text x="{x:.1f}" y="{y - R - 9:.1f}" text-anchor="middle" '
                   f'font-size="16" fill="#111">{esc(r["party"])}</text>')


    # Legend.
    lx, ly = M_L, 36
    for cat in ("left", "center", "right"):
        n = sum(1 for r in rows if r["cat"] == cat)
        out.append(f'<circle cx="{lx + 8}" cy="{ly - 5}" r="8" fill="{COLOURS[cat]}" '
                   f'stroke="#111" stroke-width="1"/>')
        out.append(f'<text x="{lx + 24}" y="{ly}" font-size="15" fill="#111">'
                   f'{cat} (n={n})</text>')
        lx += 150
    out.append("</svg>")
    return "\n".join(out)

In [ ]:
FIG_DIR.mkdir(exist_ok=True)
(FIG_DIR / 'be_party_spectrum.svg').write_text(build(parties.set_index('party').to_dict('index')))

## Pair within cycle, average per day, test across days

In [ ]:
pushes = rows.pivot_table(index=['level', 'cycle_id', 'day'], columns='arm', values='shannon')
pushes['delta'] = pushes['M'] - pushes['K']
daily = pushes.groupby(['level', 'day'])[['K', 'M', 'delta']].mean().reset_index()


def test_days(days):
    """Mean and SD of the daily values, and a one-sample t-test of the daily Δ against 0."""
    delta = days['delta']
    ttest = pg.ttest(delta, 0)
    return pd.Series({
        'K': days['K'].mean(), 'K_sd': days['K'].std(),
        'M': days['M'].mean(), 'M_sd': days['M'].std(),
        'delta': delta.mean(), 'delta_sd': delta.std(),
        't': ttest['T'].iloc[0], 'df': ttest['dof'].iloc[0], 'p': ttest['p-val'].iloc[0],
        'd_z': ttest['cohen-d'].iloc[0],            # one-sample Cohen's d = d_z
        'days_positive': (delta > 0).sum(),
    })


results = daily.groupby('level')[['K', 'M', 'delta']].apply(test_days).reindex(LEVELS)
results.round(4)

## Figure 2: daily diversity of the political part of the feeds

In [ ]:
LINES = {'K': ('Control', '#2a78d6', 'o', '-'), 'M': ('Diverse', '#eb6834', 's', '--')}

tier = daily[daily['level'] == 'tier_end'].assign(date=lambda d: pd.to_datetime(d['day']))
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for arm, (label, colour, marker, style) in LINES.items():
    ax.plot(tier['date'], tier[arm], color=colour, marker=marker, linestyle=style,
            markersize=4, linewidth=1.5, label=label)
    ax.annotate(label, (tier['date'].iloc[-1], tier[arm].iloc[-1]), xytext=(6, 0),
                textcoords='offset points', va='center', color=INK)
ax.axhline(np.log(3), color='#888888', linestyle=':', linewidth=0.8)
ax.text(tier['date'].iloc[0], np.log(3), 'Maximum (ln 3)', va='bottom', fontsize=8, color='#555555')
ax.set_ylim(0.80, 1.13)
ax.set_ylabel('Ideological diversity (Shannon H)')
ticks = pd.to_datetime(['2026-09-22', '2026-09-29', '2026-10-06', '2026-10-13', '2026-10-18'])
ax.set_xticks(ticks, [f'{d.day} {d:%b}' for d in ticks])
ax.legend(frameon=False, ncol=2, loc='lower left')

fig.savefig(FIG_DIR / 'diversity_by_day.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 3: ideological composition

Share of left, center and right votes per arm and level, with the same weighting as Δ (per cycle,
then per day, then over days). An article naming a left and a right party counts once in each.

In [ ]:
categories = ['left', 'center', 'right']
shares = rows[[f'n_{c}' for c in categories]].div(rows['votes'], axis=0).set_axis(categories, axis=1)
composition = (shares.join(rows[['level', 'arm', 'day']])
                     .groupby(['level', 'arm', 'day']).mean()
                     .groupby(['level', 'arm']).mean()
                     .reindex(LEVELS, level='level'))
(100 * composition).round(1)

In [ ]:
# Each segment carries its percentage, so the figure does not rely on colour alone.
COLOURS = {'left': ('Left', '#e34948'), 'center': ('Centre', '#a8a8a3'), 'right': ('Right', '#2a78d6')}
LEVEL_LABELS = {'top10': 'Top 10', 'top20': 'Top 20', 'tier_end': 'All political'}
ARM_LABELS = {'K': 'control', 'M': 'diversity'}

bars = composition.index.tolist()
y = [3 * i + j for i in range(len(LEVELS)) for j in range(2)]   # an empty row between levels
fig, ax = plt.subplots(figsize=(6.5, 3.2))
start = np.zeros(len(bars))
for category, (label, colour) in COLOURS.items():
    width = 100 * composition[category].to_numpy()
    ax.barh(y, width, left=start, height=0.8, color=colour, edgecolor='white', label=label)
    for row, left, w in zip(y, start, width):
        ax.text(left + w / 2, row, f'{w:.0f}%', ha='center', va='center', fontsize=8,
                color=INK if category == 'center' else 'white')
    start += width

ax.set_yticks(y, [f'{LEVEL_LABELS[level]}, {ARM_LABELS[arm]}' for level, arm in bars])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel('Share of ideology votes (%)')
ax.tick_params(axis='y', length=0)
ax.spines['left'].set_visible(False)
ax.legend(frameon=False, ncol=3, loc='lower center', bbox_to_anchor=(0.5, 1.0))

fig.savefig(FIG_DIR / 'ideology_composition.png', dpi=300, bbox_inches='tight')
plt.show()

## What diversification changed: end-of-day cycles

One cycle per day, the last one before midnight (Belgian time), describes that day's feed. Shares
are computed per day and then averaged over the 27 days.

In [ ]:
tier_end = rows[rows['level'] == 'tier_end']
completed = pd.to_datetime(tier_end['completed_at'], utc=True, format='mixed')
end_cycles = tier_end.assign(completed=completed).sort_values('completed').groupby('day')['cycle_id'].last()
day_of = pd.Series(end_cycles.index, index=end_cycles.values)          # cycle_id -> day
eod = tier_end[tier_end['cycle_id'].isin(end_cycles)]

# The cluster and source files hold exactly these cycles.
assert set(clusters['cycle_id']) == set(end_cycles) == set(pushed['cycle_id'])
pushed['day'] = pushed['cycle_id'].map(day_of)
feed_size = pushed.groupby('cycle_id').size() / 2                       # articles per feed
assert (clusters.groupby('cycle_id').size() == eod.groupby('cycle_id')['political_tier_end'].first()).all()
print(f'{len(end_cycles)} end-of-day cycles, {feed_size.iloc[0]:.0f} articles per feed')

### Articles that differ between K and M

A position differs when K and M hold different articles there, so the count is the same for both
feeds. Shares are given of the political tier (where M's selector acts) and of the whole feed.

In [ ]:
differ = eod[eod['arm'] == 'M'].set_index('day')   # n_differ is the same on the K and M rows
n_differ = differ['n_differ'] + differ['n_differ_outside_tier']
differ = pd.DataFrame({
    'n_political_tier': differ['political_tier_end'],
    'n_differ': n_differ,
    'share_of_political_tier': differ['n_differ'] / differ['political_tier_end'],
    'share_of_feed': n_differ / differ['cycle_id'].map(feed_size),
})
differ.agg(['mean', 'std', 'min', 'max']).round(3)

### One-sided clusters: positions M could not diversify

A political cluster whose articles cover at most one of left, center and right is one-sided: M has
no article from another side to choose, so that position cannot be diversified.

In [ ]:
clusters['day'] = clusters['cycle_id'].map(day_of)
clusters['one_sided'] = clusters['n_ideologies'] <= 1
one_sided_daily = clusters.groupby('day')['one_sided'].agg(n_one_sided='sum', share_of_political_tier='mean')
one_sided_daily['share_of_feed'] = one_sided_daily['n_one_sided'] / feed_size.groupby(day_of).first()
one_sided_daily.agg(['mean', 'std', 'min', 'max']).round(3)

In [ ]:
# Which side the one-sided clusters are on; 'no ideology' = political, but no party maps to a side.
one_sided = clusters[clusters['one_sided']].copy()
one_sided['side'] = np.select(
    [one_sided['n_ideologies'] == 0, one_sided['n_left'] > 0, one_sided['n_center'] > 0],
    ['no ideology', 'left', 'center'], default='right')
side_shares = (one_sided.groupby('day')['side'].value_counts(normalize=True)
                        .unstack(fill_value=0).mean()           # per day, then over days
                        .reindex(['left', 'center', 'right', 'no ideology']))
(100 * side_shares).round(1)

### Sources

Did M push some news brands more than K? Source is the domain of the linked article, within the
political tier. The total variation distance (TVD) per day is the share of political-tier
articles that would have to change source for M's source mix to equal K's (0 = identical).

In [ ]:
tier_size = pushed['cycle_id'].map(eod.groupby('cycle_id')['political_tier_end'].first())
political = pushed[pushed['rank'] <= tier_size]

source_shares = political.groupby(['arm', 'day'])['source'].value_counts(normalize=True).unstack(fill_value=0)
sources = (100 * source_shares.groupby('arm').mean().T).rename(columns={'K': 'K_%', 'M': 'M_%'})
sources['M_minus_K'] = sources['M_%'] - sources['K_%']

tvd = 0.5 * (source_shares.xs('M') - source_shares.xs('K')).abs().sum(axis=1)
print(f'Source mix, M vs K: TVD = {tvd.mean():.3f} (SD {tvd.std():.3f}, range {tvd.min():.3f}-{tvd.max():.3f})')
sources.sort_values('K_%', ascending=False).round(1)